In [0]:
STORAGE_ACCOUNT = "vjretailflow01"
STORAGE_KEY = dbutils.secrets.get(scope="my-scope", key="storage-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

print("Storage Account Key Set")

In [0]:
BASE_PATH = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/olist/2024-01-01/"
BRONZE_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"

SOURCES = {
    "orders":             f"{BASE_PATH}/orders/",
    "order_items":        f"{BASE_PATH}/order_items/",
    "payments":           f"{BASE_PATH}/payments/",
    "reviews":            f"{BASE_PATH}/reviews/",
    "customers":          f"{BASE_PATH}/customers/",
    "products":           f"{BASE_PATH}/products/",
    "sellers":            f"{BASE_PATH}/sellers/",
    "geolocation":        f"{BASE_PATH}/geolocation/",
    "category":           f"{BASE_PATH}/category/",
}

print("Paths configured")

In [0]:
dbutils.fs.ls(f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/olist/2024-01-01/")

In [0]:
from pyspark.sql import functions as F
from datetime import datetime
from pyspark.sql import types as T

In [0]:
INGESTION_DATE = "2024-01-01"

def ingest_to_bronze(source_name, source_path, bronze_path):
    print(f"Ingesting {source_name}...")

    df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(source_path)

    df = df.withColumn("_ingested_at", F.current_timestamp()) \
            .withColumn("_source_file", F.lit(source_path)) \
            .withColumn("_ingestion_date", F.lit(INGESTION_DATE))

    df.write.format("delta") \
        .mode("overwrite") \
        .save(f"{bronze_path}/{source_name}")
    
    count = df.count()
    print(f"{source_name}: {count} rows written to bronze")
    return count

results = {}
for source_name, source_path in SOURCES.items():
    results[source_name] = ingest_to_bronze(source_name, source_path, BRONZE_PATH)

print("\n Ingestion Summary")
for name, count in results.items():
    print(f"{name:20s}: {count:,} rows")

In [0]:
orders_bronze = spark.read.format("delta") \
                    .load(f"{BRONZE_PATH}/orders/")

print(f"Schema:")
orders_bronze.printSchema()

print(f"\n Sample rows")
orders_bronze.show(3, truncate=False)

print(f"\n Audit columns present")
audit_cols = ["_ingested_at", "_source_file", "_ingestion_date"]
orders_bronze.select(audit_cols).show(3)